# 03. SQL Analysis and Tableau Data Preparation

У цьому ноутбуці виконую SQL-аналіз через `DuckDB` та готую фінальні CSV-файли для Tableau dashboard.

Для аналізу використовую підготовлені файли `app_clean.csv` та `prev_clean.csv`, які були створені в ноутбуці `01_data_preparation.ipynb`.

In [1]:
import pandas as pd
import duckdb

from pathlib import Path

In [2]:
# Завантаження підготовлених даних

app_clean = pd.read_csv("../data/processed/app_clean.csv")
prev_clean = pd.read_csv("../data/processed/prev_clean.csv")

print(f"app_clean shape: {app_clean.shape}")
print(f"prev_clean shape: {prev_clean.shape}")

app_clean shape: (307511, 129)
prev_clean shape: (1670214, 37)


## 8. SQL analysis with DuckDB

Використовую `DuckDB`, щоб виконати кілька SQL-запитів без створення окремої бази даних.

Частину результатів EDA перевіряю через SQL, а також додаю нові зрізи з використанням `JOIN`.

Ключові питання:

1. Як змінюється `default rate` залежно від кредитного навантаження клієнта?
2. Чи пов’язана кількість попередніх заявок клієнта з дефолтом?
3. Які комбінації статусу та типу попереднього продукту мають найвищий `default rate`?

In [3]:
# Підключення pandas DataFrames до DuckDB

con = duckdb.connect()

con.register("app_clean", app_clean)
con.register("prev_clean", prev_clean)

### 8.1. Як змінюється default rate залежно від кредитного навантаження?

У цьому запиті перевіряю, як змінюється `default rate` залежно від групи `credit_income_group`.

In [4]:
query = """
SELECT
    credit_income_group,
    COUNT(*) AS clients_count,
    SUM(TARGET) AS default_clients,
    ROUND(AVG(TARGET) * 100, 2) AS default_rate
FROM app_clean
WHERE credit_income_group IS NOT NULL
GROUP BY credit_income_group
ORDER BY 
    CASE credit_income_group
        WHEN '0-1' THEN 1
        WHEN '1-2' THEN 2
        WHEN '2-3' THEN 3
        WHEN '3-5' THEN 4
        WHEN '5-10' THEN 5
        WHEN '10+' THEN 6
    END
"""

sql_credit_income = con.execute(query).df()
sql_credit_income

,credit_income_group,clients_count,default_clients,default_rate
0,0-1,14049,920.0,6.55
1,1-2,56439,4348.0,7.70
2,2-3,66165,5747.0,8.69
3,3-5,86751,7612.0,8.77
4,5-10,73070,5471.0,7.49
5,10+,11037,727.0,6.59


SQL-запит підтверджує результати EDA: `default rate` відрізняється між групами кредитного навантаження.

Найвищі значення спостерігаються не у крайніх групах, а в середніх діапазонах `credit_income_ratio` — `2-3` та `3-5`. Це означає, що зв’язок між кредитним навантаженням і дефолтом не є повністю лінійним.

### 8.2. Чи пов’язана кількість попередніх заявок клієнта з дефолтом?

На цьому етапі перевіряю, чи змінюється `default rate` залежно від кількості попередніх заявок клієнта.

Оскільки один клієнт може мати кілька попередніх заявок, спочатку агрегую `prev_clean` на рівні клієнта, а потім через `LEFT JOIN` приєдную результат до `app_clean`.

In [5]:
con.register("prev_clean", prev_clean)

In [6]:
query = """
WITH prev_counts AS (
    SELECT
        SK_ID_CURR,
        COUNT(SK_ID_PREV) AS previous_applications_count
    FROM prev_clean
    GROUP BY SK_ID_CURR
),

app_with_prev AS (
    SELECT
        a.SK_ID_CURR,
        a.TARGET,
        COALESCE(p.previous_applications_count, 0) AS previous_applications_count
    FROM app_clean AS a
    LEFT JOIN prev_counts AS p
        ON a.SK_ID_CURR = p.SK_ID_CURR
),

prev_groups AS (
    SELECT
        CASE
            WHEN previous_applications_count = 0 THEN '0'
            WHEN previous_applications_count = 1 THEN '1'
            WHEN previous_applications_count BETWEEN 2 AND 3 THEN '2-3'
            WHEN previous_applications_count BETWEEN 4 AND 5 THEN '4-5'
            ELSE '6+'
        END AS previous_applications_group,
        CASE
            WHEN previous_applications_count = 0 THEN 1
            WHEN previous_applications_count = 1 THEN 2
            WHEN previous_applications_count BETWEEN 2 AND 3 THEN 3
            WHEN previous_applications_count BETWEEN 4 AND 5 THEN 4
            ELSE 5
        END AS sort_order,
        TARGET
    FROM app_with_prev
)

SELECT
    previous_applications_group,
    COUNT(*) AS clients_count,
    CAST(SUM(TARGET) AS INTEGER) AS default_clients,
    ROUND(AVG(TARGET) * 100, 2) AS default_rate
FROM prev_groups
GROUP BY previous_applications_group, sort_order
ORDER BY sort_order
"""

sql_prev_count = con.execute(query).df()
sql_prev_count

,previous_applications_group,clients_count,default_clients,default_rate
0,0,16454,980,5.96
1,1,52533,4400,8.38
2,2-3,85813,6725,7.84
3,4-5,59560,4588,7.70
4,6+,93151,8132,8.73


**Висновок:**

SQL-запит із `LEFT JOIN` показує, що `default rate` відрізняється залежно від кількості попередніх заявок клієнта.

Клієнти без попередніх заявок мають найнижчий `default rate` — **5.96%**.

Для клієнтів з однією попередньою заявкою показник зростає до **8.38%**, а для клієнтів із 6+ попередніми заявками становить **8.73%**.

Зв’язок не є повністю лінійним, але загалом клієнти з історією попередніх заявок мають вищий рівень дефолту, ніж клієнти без такої історії.

Ця ознака може бути корисною для подальшого аналізу та ML-блоку.

### 8.3. Які комбінації статусу та типу попереднього продукту мають найвищий default rate?

На цьому етапі через SQL `JOIN` поєдную основну таблицю `app_clean` з таблицею попередніх заявок `prev_clean`.

Мета — перевірити, які комбінації статусу попередньої заявки та типу попереднього кредитного продукту мають найвищий `default rate`.

Для стабільнішої інтерпретації залишаю лише групи з достатньою кількістю попередніх заявок.

Таблицю сортую за `default_rate` у спадному порядку, щоб одразу побачити комбінації з найвищим ризиком дефолту.

In [7]:
query = """
SELECT
    p.NAME_CONTRACT_STATUS AS previous_status,
    p.NAME_CONTRACT_TYPE AS previous_contract_type,
    COUNT(p.SK_ID_PREV) AS applications_count,
    CAST(SUM(a.TARGET) AS INTEGER) AS default_applications,
    ROUND(AVG(a.TARGET) * 100, 2) AS default_rate
FROM prev_clean AS p
INNER JOIN app_clean AS a
    ON p.SK_ID_CURR = a.SK_ID_CURR
WHERE p.NAME_CONTRACT_TYPE != 'XNA'
GROUP BY
    p.NAME_CONTRACT_STATUS,
    p.NAME_CONTRACT_TYPE
HAVING COUNT(p.SK_ID_PREV) >= 1000
ORDER BY default_rate DESC
"""

sql_status_product = con.execute(query).df()
sql_status_product

,previous_status,previous_contract_type,applications_count,default_applications,default_rate
0,Refused,Revolving loans,41511,5357,12.91
1,Canceled,Consumer loans,1329,171,12.87
2,Refused,Cash loans,139568,17559,12.58
3,Canceled,Revolving loans,37445,4091,10.93
4,Refused,Consumer loans,64282,6515,10.14
5,Approved,Revolving loans,82408,7445,9.03
6,Canceled,Cash loans,220383,19482,8.84
7,Unused offer,Consumer loans,22335,1839,8.23
8,Approved,Cash loans,266381,20116,7.55
9,Approved,Consumer loans,537310,39682,7.39


**Висновок:**

Комбінований SQL-зріз показує, що найвищий `default rate` мають попередні заявки зі статусом `Refused` та типом продукту `Revolving loans` — **12.91%**.

Також високий рівень дефолту спостерігається для комбінацій:

- `Canceled` + `Consumer loans` — **12.87%**;
- `Refused` + `Cash loans` — **12.58%**;
- `Canceled` + `Revolving loans` — **10.93%**.

Найнижчі значення серед основних комбінацій мають попередні заявки зі статусом `Approved`, особливо для `Consumer loans` — **7.39%**.

Отже, комбінація статусу попередньої заявки та типу попереднього продукту може бути корисною для оцінки ризику дефолту.

**Загальний висновок по SQL analysis:**

SQL-запити підтвердили частину результатів EDA та дозволили додати нові зрізи через `JOIN`.

Кредитне навантаження, кількість попередніх заявок, статус попередньої заявки та тип попереднього продукту мають зв’язок із `default rate`.

Найбільш ризиковими виглядають клієнти з історією попередніх заявок, особливо якщо в минулому були відмовлені або скасовані заявки, а також продукти типу `Revolving loans`.

Ці результати можна використати для формування бізнес-висновків і як потенційні ознаки для ML-блоку.

### 8.4. Підготовка даних для Tableau dashboard

На цьому етапі готую фінальні CSV-файли для Tableau.

Потрібні ознаки вже створені раніше у блоках `Feature engineering`, `EDA` та `Previous applications analysis`, тому тут не дублюю підготовку даних, а лише формую готові таблиці для візуалізації.

In [8]:
# Client-level dataset for Tableau

query = """
WITH prev_summary AS (
    SELECT
        SK_ID_CURR,
        COUNT(SK_ID_PREV) AS previous_applications_count,

        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END) AS refused_previous_count,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS approved_previous_count,

        SUM(CASE WHEN NAME_CONTRACT_TYPE = 'Revolving loans' THEN 1 ELSE 0 END) AS revolving_loans_previous_count,

        AVG(
            CASE 
                WHEN NAME_CONTRACT_STATUS = 'Approved'
                     AND AMT_APPLICATION > 0
                     AND AMT_CREDIT IS NOT NULL
                THEN AMT_CREDIT - AMT_APPLICATION
            END
        ) AS avg_offer_gap

    FROM prev_clean
    GROUP BY SK_ID_CURR
)

SELECT
    a.SK_ID_CURR,
    a.TARGET,
    CASE WHEN a.TARGET = 1 THEN 'Default' ELSE 'No default' END AS target_label,

    a.CODE_GENDER,

    CASE
        WHEN a.CODE_GENDER = 'F' THEN 'Female'
        WHEN a.CODE_GENDER = 'M' THEN 'Male'
        ELSE 'Unknown'
    END AS gender_label,

    a.age_years,
    a.age_group,
    a.employment_years,

    a.AMT_INCOME_TOTAL,
    a.AMT_CREDIT,
    a.AMT_ANNUITY,

    a.credit_income_ratio,
    a.credit_income_group,
    a.annuity_income_ratio,

    a.NAME_INCOME_TYPE,
    a.NAME_EDUCATION_TYPE,
    a.NAME_FAMILY_STATUS,
    a.children_group,

    COALESCE(p.previous_applications_count, 0) AS previous_applications_count,

    CASE
        WHEN COALESCE(p.previous_applications_count, 0) = 0 THEN '0'
        WHEN COALESCE(p.previous_applications_count, 0) = 1 THEN '1'
        WHEN COALESCE(p.previous_applications_count, 0) BETWEEN 2 AND 3 THEN '2-3'
        WHEN COALESCE(p.previous_applications_count, 0) BETWEEN 4 AND 5 THEN '4-5'
        ELSE '6+'
    END AS previous_applications_group,

    CASE 
        WHEN COALESCE(p.refused_previous_count, 0) > 0 THEN 'Has refused previous application'
        ELSE 'No refused previous application'
    END AS has_refused_previous_application,

    CASE 
        WHEN COALESCE(p.revolving_loans_previous_count, 0) > 0 THEN 'Has revolving loans'
        ELSE 'No revolving loans'
    END AS has_revolving_previous_product,

    p.avg_offer_gap,

    CASE
        WHEN p.avg_offer_gap IS NULL THEN 'No approved previous application'
        WHEN p.avg_offer_gap < 0 THEN 'Credit lower than requested'
        WHEN p.avg_offer_gap = 0 THEN 'Credit equal to requested'
        WHEN p.avg_offer_gap > 0 THEN 'Credit higher than requested'
    END AS avg_offer_gap_group

FROM app_clean AS a
LEFT JOIN prev_summary AS p
    ON a.SK_ID_CURR = p.SK_ID_CURR
"""

dashboard_client_data = con.execute(query).df()

dashboard_client_data.head()

,SK_ID_CURR,TARGET,target_label,CODE_GENDER,gender_label,age_years,age_group,employment_years,AMT_INCOME_TOTAL,AMT_CREDIT,...,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,children_group,previous_applications_count,previous_applications_group,has_refused_previous_application,has_revolving_previous_product,avg_offer_gap,avg_offer_gap_group
0,125792,0,No default,F,Female,44.5,40-49,2.1,360000.0,900000.0,...,Working,Higher education,Widow,No children,1,1,No refused previous application,No revolving loans,0.000,Credit equal to requested
1,143722,0,No default,F,Female,43.7,40-49,20.5,202500.0,501435.0,...,Working,Secondary / secondary special,Married,Has children,6,6+,No refused previous application,Has revolving loans,18177.000,Credit higher than requested
2,363365,0,No default,F,Female,61.6,60-69,0.3,139500.0,288873.0,...,Working,Lower secondary,Separated,No children,7,6+,Has refused previous application,No revolving loans,-2002.500,Credit lower than requested
3,370678,1,Default,F,Female,39.1,30-39,0.5,157500.0,490536.0,...,Working,Secondary / secondary special,Separated,No children,3,2-3,No refused previous application,No revolving loans,1602.000,Credit higher than requested
4,111543,0,No default,M,Male,51.7,50-59,17.9,247500.0,1241437.5,...,Commercial associate,Secondary / secondary special,Married,No children,9,6+,Has refused previous application,No revolving loans,6235.875,Credit higher than requested


In [9]:
# Previous-application-level dataset for Tableau

query = """
SELECT
    p.SK_ID_PREV,
    p.SK_ID_CURR,
    a.TARGET,
    CASE WHEN a.TARGET = 1 THEN 'Default' ELSE 'No default' END AS target_label,

    p.NAME_CONTRACT_STATUS,
    p.NAME_CONTRACT_TYPE,
    CASE 
        WHEN p.NAME_CONTRACT_TYPE = 'XNA' THEN 'Unknown'
        ELSE p.NAME_CONTRACT_TYPE
    END AS contract_type_clean,

    p.NAME_CLIENT_TYPE,
    p.NAME_PRODUCT_TYPE,
    p.CHANNEL_TYPE,

    p.AMT_APPLICATION,
    p.AMT_CREDIT,

    CASE 
        WHEN p.NAME_CONTRACT_STATUS = 'Approved'
             AND p.AMT_APPLICATION > 0
             AND p.AMT_CREDIT IS NOT NULL
        THEN p.AMT_CREDIT - p.AMT_APPLICATION
    END AS offer_gap,

    CASE
        WHEN p.NAME_CONTRACT_STATUS != 'Approved' THEN 'Not approved'
        WHEN p.AMT_APPLICATION IS NULL OR p.AMT_CREDIT IS NULL OR p.AMT_APPLICATION <= 0 THEN 'No amount data'
        WHEN p.AMT_CREDIT - p.AMT_APPLICATION < 0 THEN 'Credit lower than requested'
        WHEN p.AMT_CREDIT - p.AMT_APPLICATION = 0 THEN 'Credit equal to requested'
        WHEN p.AMT_CREDIT - p.AMT_APPLICATION > 0 THEN 'Credit higher than requested'
    END AS offer_gap_group

FROM prev_clean AS p
INNER JOIN app_clean AS a
    ON p.SK_ID_CURR = a.SK_ID_CURR
"""

dashboard_previous_application_data = con.execute(query).df()

dashboard_previous_application_data.head()

,SK_ID_PREV,SK_ID_CURR,TARGET,target_label,NAME_CONTRACT_STATUS,NAME_CONTRACT_TYPE,contract_type_clean,NAME_CLIENT_TYPE,NAME_PRODUCT_TYPE,CHANNEL_TYPE,AMT_APPLICATION,AMT_CREDIT,offer_gap,offer_gap_group
0,1461222,134253,0,No default,Approved,Consumer loans,Consumer loans,Repeater,XNA,Stone,49275.0,49275.0,0.0,Credit equal to requested
1,1495120,205272,0,No default,Canceled,Cash loans,Cash loans,Repeater,XNA,Credit and cash offices,0.0,0.0,NaN,Not approved
2,2819834,157817,0,No default,Approved,Cash loans,Cash loans,Repeater,x-sell,Credit and cash offices,1129500.0,1244304.0,114804.0,Credit higher than requested
3,2666631,260772,0,No default,Canceled,Cash loans,Cash loans,Repeater,XNA,Credit and cash offices,0.0,0.0,NaN,Not approved
4,2541914,304247,0,No default,Approved,Consumer loans,Consumer loans,Repeater,XNA,Country-wide,130455.0,104364.0,-26091.0,Credit lower than requested


In [10]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

dashboard_client_data.to_csv(
    processed_path / "dashboard_client_data.csv",
    index=False,
    encoding="utf-8-sig"
)

dashboard_previous_application_data.to_csv(
    processed_path / "dashboard_previous_application_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dashboard CSV files saved successfully.")

Dashboard CSV files saved successfully.


Для Tableau підготовлено два CSV-файли:

- `dashboard_client_data.csv` — дані на рівні клієнта;
- `dashboard_previous_application_data.csv` — дані на рівні попередньої заявки.

Ці файли будуть використані для побудови Tableau dashboards.

## Summary

У цьому ноутбуці SQL-запити через `DuckDB` були використані для перевірки кількох бізнес-питань та підготовки даних для Tableau.

Основні результати:

- `default rate` відрізняється між групами кредитного навантаження;
- клієнти з історією попередніх заявок мають інший рівень дефолту порівняно з клієнтами без такої історії;
- комбінації статусу та типу попереднього кредитного продукту допомагають краще оцінити ризикові сегменти;
- підготовлено два CSV-файли для Tableau: `dashboard_client_data.csv` та `dashboard_previous_application_data.csv`.

CSV-файли збережено локально в `data/processed`, але вони не завантажуються в GitHub.